# Fraud Detection Guided Notebook (Same Preprocessing Ensemble)

동일 전처리를 공유한 LightGBM + CatBoost + XGBoost 앙상블 학습 파이프라인입니다.

## 1) 환경 설정 및 라이브러리 임포트

In [1]:
import json
import random
import warnings
from pathlib import Path

import joblib
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

USE_OPTUNA = True
N_SPLITS = 5
N_REPEATS_TUNE = 2
N_TRIALS_PER_MODEL = 40
N_TRIALS_ENSEMBLE = 300

DATA_DIR = Path(".")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

print("seed:", SEED)
print("use_optuna:", USE_OPTUNA)
print("n_splits:", N_SPLITS)
print("n_repeats_tune:", N_REPEATS_TUNE)
print("trials per model:", N_TRIALS_PER_MODEL)
print("ensemble trials:", N_TRIALS_ENSEMBLE)

seed: 42
use_optuna: True
n_splits: 5
n_repeats_tune: 2
trials per model: 40
ensemble trials: 300


## 2) 데이터 로드 및 기본 검증

In [2]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

print("train shape:", train_raw.shape)
print("test shape :", test_raw.shape)
print("target mean:", train_raw["fraud_bool"].mean())
print("class counts:")
print(train_raw["fraud_bool"].value_counts())
print("\nTop missing columns (train):")
print(train_raw.isna().mean().sort_values(ascending=False).head(12))

train shape: (700000, 33)
test shape : (300000, 32)
target mean: 0.010917142857142858
class counts:
fraud_bool
0    692358
1      7642
Name: count, dtype: int64

Top missing columns (train):
id                   0.0
fraud_bool           0.0
yearly_income        0.0
name_email_sim       0.0
prev_addr_months     0.0
curr_addr_months     0.0
age_bucket           0.0
days_since_req       0.0
init_transfer_amt    0.0
payment_type         0.0
zip_req_count_4w     0.0
req_rate_6h          0.0
dtype: float64


## 3) 권장 전처리 파이프라인 정의 (업데이트)

- 무의미 변수 제거 (`id`, `device_fraud_history`)
- 센티넬(-1) 결측 플래그 생성
- MI 상위 이진 피처 유지 + 예외 패턴 플래그 강화
- `req_rate_*` 3개 동시 사용 + 비율 파생 + 다봉 분포 binning + 이상치 변환
- `month_idx`는 계절성 범주로 처리(원-핫 + sin/cos)
- 단순 절대 임계값보다 `age_bucket` 조건부 상대 피처(차이/비율/Z-score) 중심
- 원본 변수 + 연령대 상대 피처를 함께 투입해 트리 모델이 비선형 상호작용 학습하도록 구성

In [3]:
def merge_rare_categories(train_s: pd.Series, test_s: pd.Series, min_freq: float = 0.005):
    freq = train_s.value_counts(normalize=True)
    rare = set(freq[freq < min_freq].index)
    tr = train_s.where(~train_s.isin(rare), "Other")
    te = test_s.where(~test_s.isin(rare), "Other")
    return tr, te


def clip_log_transform(train_df: pd.DataFrame, test_df: pd.DataFrame, col: str, q_low=0.0, q_high=0.995):
    lo = float(train_df[col].quantile(q_low))
    hi = float(train_df[col].quantile(q_high))
    train_df[col] = np.log1p(train_df[col].clip(lo, hi))
    test_df[col] = np.log1p(test_df[col].clip(lo, hi))


def preprocess_guided(train_df: pd.DataFrame, test_df: pd.DataFrame):
    train = train_df.copy()
    test = test_df.copy()

    # remove uninformative and id-like columns
    drop_base = ["id", "device_fraud_history"]
    for c in drop_base:
        if c in train.columns:
            train = train.drop(columns=c)
        if c in test.columns:
            test = test.drop(columns=c)

    # keep MI-top binary features as-is, but normalize dtype
    mi_top_bin = [
        "device_email_cnt_8w",
        "is_session_persistent",
        "is_mobile_valid",
        "is_free_email",
        "is_home_phone_valid",
    ]
    for c in mi_top_bin:
        if c in train.columns:
            train[c] = pd.to_numeric(train[c], errors="coerce")
            test[c] = pd.to_numeric(test[c], errors="coerce")

    # sentinel(-1) handling with missing flags
    for c in ["prev_addr_months", "bank_months_count"]:
        if c in train.columns:
            med = train.loc[train[c] != -1, c].median()
            train[f"is_missing_{c}"] = (train[c] == -1).astype(np.int8)
            test[f"is_missing_{c}"] = (test[c] == -1).astype(np.int8)
            train[c] = train[c].where(train[c] != -1, med)
            test[c] = test[c].where(test[c] != -1, med)

    # MI insight: device_email_cnt_8w exceptional values are highly informative
    if "device_email_cnt_8w" in train.columns:
        tr_dev = pd.to_numeric(train["device_email_cnt_8w"], errors="coerce")
        te_dev = pd.to_numeric(test["device_email_cnt_8w"], errors="coerce")
        train["flag_device_email_neq1"] = (tr_dev != 1).astype(np.int8)
        test["flag_device_email_neq1"] = (te_dev != 1).astype(np.int8)
        train["flag_device_email_missing"] = (tr_dev.isna() | (tr_dev == -1)).astype(np.int8)
        test["flag_device_email_missing"] = (te_dev.isna() | (te_dev == -1)).astype(np.int8)
        med_dev = tr_dev.replace(-1, np.nan).median()
        train["device_email_cnt_8w"] = tr_dev.replace(-1, np.nan).fillna(med_dev)
        test["device_email_cnt_8w"] = te_dev.replace(-1, np.nan).fillna(med_dev)

    # base binary fraud flags
    if "name_email_sim" in train.columns:
        train["flag_name_email_low"] = (train["name_email_sim"] < 0.3).astype(np.int8)
        test["flag_name_email_low"] = (test["name_email_sim"] < 0.3).astype(np.int8)
        train["name_email_sim"] = train["name_email_sim"].clip(0, 1)
        test["name_email_sim"] = test["name_email_sim"].clip(0, 1)

    # update: avoid global thresholding for credit_risk_score/req_credit_limit; keep only basic sanitation
    if "credit_risk_score" in train.columns:
        train["credit_risk_score"] = pd.to_numeric(train["credit_risk_score"], errors="coerce")
        test["credit_risk_score"] = pd.to_numeric(test["credit_risk_score"], errors="coerce")
        cr_med = train["credit_risk_score"].replace(-1, np.nan).median()
        train["credit_risk_score"] = train["credit_risk_score"].replace(-1, np.nan).fillna(cr_med)
        test["credit_risk_score"] = test["credit_risk_score"].replace(-1, np.nan).fillna(cr_med)

    if "req_credit_limit" in train.columns:
        train["req_credit_limit"] = pd.to_numeric(train["req_credit_limit"], errors="coerce")
        test["req_credit_limit"] = pd.to_numeric(test["req_credit_limit"], errors="coerce")
        rcl_med = train["req_credit_limit"].median()
        train["req_credit_limit"] = train["req_credit_limit"].fillna(rcl_med)
        test["req_credit_limit"] = test["req_credit_limit"].fillna(rcl_med)

    if "days_since_req" in train.columns:
        train["flag_days_since_req_lt7"] = (train["days_since_req"] < 7).astype(np.int8)
        test["flag_days_since_req_lt7"] = (test["days_since_req"] < 7).astype(np.int8)

    if "init_transfer_amt" in train.columns:
        threshold = float(train.loc[train["init_transfer_amt"] >= 0, "init_transfer_amt"].quantile(0.95))
        train["flag_init_transfer_high"] = (train["init_transfer_amt"] >= threshold).astype(np.int8)
        test["flag_init_transfer_high"] = (test["init_transfer_amt"] >= threshold).astype(np.int8)
        train["init_transfer_amt"] = train["init_transfer_amt"].clip(lower=0)
        test["init_transfer_amt"] = test["init_transfer_amt"].clip(lower=0)

    # yearly income U-shape flags
    if "yearly_income" in train.columns:
        y_lo = float(train["yearly_income"].quantile(0.1))
        y_hi = float(train["yearly_income"].quantile(0.9))
        train["flag_income_low"] = (train["yearly_income"] <= y_lo).astype(np.int8)
        train["flag_income_high"] = (train["yearly_income"] >= y_hi).astype(np.int8)
        test["flag_income_low"] = (test["yearly_income"] <= y_lo).astype(np.int8)
        test["flag_income_high"] = (test["yearly_income"] >= y_hi).astype(np.int8)

    # high risk category flags
    mapping_flags = [
        ("housing_status", "BA", "flag_housing_BA"),
        ("device_os", "windows", "flag_os_windows"),
        ("employment_status", "CC", "flag_employment_CC"),
        ("payment_type", "AC", "flag_payment_AC"),
        ("application_source", "TELEAPP", "flag_source_TELEAPP"),
    ]
    for col, val, out_col in mapping_flags:
        if col in train.columns:
            train[out_col] = (train[col].astype(str) == val).astype(np.int8)
            test[out_col] = (test[col].astype(str) == val).astype(np.int8)

    # month_idx as seasonal categorical + cyclic encoding
    if "month_idx" in train.columns:
        nm = 8.0
        m_tr = pd.to_numeric(train["month_idx"], errors="coerce").fillna(0.0)
        m_te = pd.to_numeric(test["month_idx"], errors="coerce").fillna(0.0)
        train["month_sin"] = np.sin(2 * np.pi * m_tr / nm).astype(np.float32)
        train["month_cos"] = np.cos(2 * np.pi * m_tr / nm).astype(np.float32)
        test["month_sin"] = np.sin(2 * np.pi * m_te / nm).astype(np.float32)
        test["month_cos"] = np.cos(2 * np.pi * m_te / nm).astype(np.float32)
        train["month_idx_cat"] = train["month_idx"].astype(str)
        test["month_idx_cat"] = test["month_idx"].astype(str)
        train = train.drop(columns=["month_idx"])
        test = test.drop(columns=["month_idx"])

    # req_rate_* specialized processing for tree models
    req_cols = ["req_rate_6h", "req_rate_24h", "req_rate_4w"]
    has_req = all(c in train.columns for c in req_cols)
    if has_req:
        tr6_raw = train["req_rate_6h"].clip(lower=0)
        tr24_raw = train["req_rate_24h"].clip(lower=0)
        tr4w_raw = train["req_rate_4w"].clip(lower=0)
        te6_raw = test["req_rate_6h"].clip(lower=0)
        te24_raw = test["req_rate_24h"].clip(lower=0)
        te4w_raw = test["req_rate_4w"].clip(lower=0)

        train["feat_req24_over_4w"] = tr24_raw / (tr4w_raw + 1e-6)
        test["feat_req24_over_4w"] = te24_raw / (te4w_raw + 1e-6)
        train["feat_req6_over_24"] = tr6_raw / (tr24_raw + 1e-6)
        test["feat_req6_over_24"] = te6_raw / (te24_raw + 1e-6)
        train["feat_req6_over_4w"] = tr6_raw / (tr4w_raw + 1e-6)
        test["feat_req6_over_4w"] = te6_raw / (te4w_raw + 1e-6)

        q4 = np.unique(tr4w_raw.quantile([0.0, 1 / 3, 2 / 3, 1.0]).values)
        if len(q4) >= 4:
            train["req_rate_4w_bin3"] = pd.cut(tr4w_raw, bins=q4, include_lowest=True, labels=False)
            test["req_rate_4w_bin3"] = pd.cut(te4w_raw, bins=q4, include_lowest=True, labels=False)
        else:
            train["req_rate_4w_bin3"] = pd.qcut(tr4w_raw.rank(method="first"), 3, labels=False)
            test["req_rate_4w_bin3"] = pd.cut(te4w_raw, bins=np.quantile(tr4w_raw, [0, 1 / 3, 2 / 3, 1]), include_lowest=True, labels=False)
        train["req_rate_4w_bin3"] = train["req_rate_4w_bin3"].fillna(0).astype(np.int8)
        test["req_rate_4w_bin3"] = test["req_rate_4w_bin3"].fillna(0).astype(np.int8)

        med24 = float(tr24_raw.median())
        train["req_rate_24h_bin2"] = (tr24_raw >= med24).astype(np.int8)
        test["req_rate_24h_bin2"] = (te24_raw >= med24).astype(np.int8)

    # age-conditioned relative features (core update)
    if "age_bucket" in train.columns:
        age_tr = train["age_bucket"].astype(str)
        age_te = test["age_bucket"].astype(str)

        rel_cols = [
            "credit_risk_score",
            "req_credit_limit",
            "curr_addr_months",
            "name_email_sim",
            "session_length_min",
        ]
        for c in rel_cols:
            if c in train.columns:
                tr_num = pd.to_numeric(train[c], errors="coerce")
                te_num = pd.to_numeric(test[c], errors="coerce")

                grp_mean = tr_num.groupby(age_tr).mean()
                grp_std = tr_num.groupby(age_tr).std().replace(0, np.nan)
                global_mean = float(tr_num.mean())
                global_std = float(tr_num.std()) if float(tr_num.std()) > 1e-6 else 1.0

                tr_mu = age_tr.map(grp_mean).fillna(global_mean)
                te_mu = age_te.map(grp_mean).fillna(global_mean)
                tr_sd = age_tr.map(grp_std).fillna(global_std)
                te_sd = age_te.map(grp_std).fillna(global_std)

                train[f"age_adj_{c}_diff"] = (tr_num - tr_mu).astype(np.float32)
                test[f"age_adj_{c}_diff"] = (te_num - te_mu).astype(np.float32)

                train[f"age_adj_{c}_ratio"] = (tr_num / (tr_mu + 1e-6)).astype(np.float32)
                test[f"age_adj_{c}_ratio"] = (te_num / (te_mu + 1e-6)).astype(np.float32)

                train[f"age_adj_{c}_z"] = ((tr_num - tr_mu) / (tr_sd + 1e-6)).astype(np.float32)
                test[f"age_adj_{c}_z"] = ((te_num - te_mu) / (te_sd + 1e-6)).astype(np.float32)

                # optional anomaly flags based on age-conditioned z-score
                train[f"age_adj_{c}_absz_gt2"] = (np.abs(train[f"age_adj_{c}_z"]) > 2.0).astype(np.int8)
                test[f"age_adj_{c}_absz_gt2"] = (np.abs(test[f"age_adj_{c}_z"]) > 2.0).astype(np.int8)

    # skewed variable clip + log1p (including req_rate_*)
    skew_cols = [
        "curr_addr_months", "session_length_min", "req_credit_limit", "branch_req_count_8w",
        "days_since_req", "dob_email_count_4w", "init_transfer_amt", "zip_req_count_4w",
        "req_rate_6h", "req_rate_24h", "req_rate_4w",
    ]
    for c in skew_cols:
        if c in train.columns:
            clip_log_transform(train, test, c, q_low=0.0, q_high=0.995)

    for c in ["req_rate_4w", "req_rate_24h"]:
        if c in train.columns and f"{c}_bin" not in train.columns:
            edges = np.unique(train[c].quantile([0.0, 0.33, 0.66, 1.0]).values)
            if len(edges) >= 3:
                train[f"{c}_bin"] = pd.cut(train[c], bins=edges, include_lowest=True, labels=False).astype("float")
                test[f"{c}_bin"] = pd.cut(test[c], bins=edges, include_lowest=True, labels=False).astype("float")
                train[f"{c}_bin"] = train[f"{c}_bin"].fillna(0).astype(np.int8)
                test[f"{c}_bin"] = test[f"{c}_bin"].fillna(0).astype(np.int8)

    # interaction features
    if "req_credit_limit" in train.columns and "credit_risk_score" in train.columns:
        train["feat_limit_over_credit"] = train["req_credit_limit"] / (train["credit_risk_score"] + 1e-3)
        test["feat_limit_over_credit"] = test["req_credit_limit"] / (test["credit_risk_score"] + 1e-3)

    if "credit_risk_score" in train.columns and "age_bucket" in train.columns:
        age_tr_num = pd.to_numeric(train["age_bucket"], errors="coerce").fillna(0)
        age_te_num = pd.to_numeric(test["age_bucket"], errors="coerce").fillna(0)
        train["feat_credit_x_age"] = train["credit_risk_score"] * age_tr_num
        test["feat_credit_x_age"] = test["credit_risk_score"] * age_te_num

    if "curr_addr_months" in train.columns and "name_email_sim" in train.columns:
        train["feat_curr_x_name_sim"] = train["curr_addr_months"] * train["name_email_sim"]
        test["feat_curr_x_name_sim"] = test["curr_addr_months"] * test["name_email_sim"]

    # MI insight: explicit interactions between strong binary signals and continuous features
    if "name_email_sim" in train.columns and "is_mobile_valid" in train.columns:
        train["feat_name_x_mobile_valid"] = train["name_email_sim"] * train["is_mobile_valid"]
        test["feat_name_x_mobile_valid"] = test["name_email_sim"] * test["is_mobile_valid"]

    if "credit_risk_score" in train.columns and "is_session_persistent" in train.columns:
        train["feat_credit_x_session_persistent"] = train["credit_risk_score"] * train["is_session_persistent"]
        test["feat_credit_x_session_persistent"] = test["credit_risk_score"] * test["is_session_persistent"]

    # rare category merge for key columns
    for c in ["housing_status", "device_os", "employment_status", "payment_type", "application_source"]:
        if c in train.columns:
            train[c], test[c] = merge_rare_categories(train[c].astype(str), test[c].astype(str), min_freq=0.005)

    # one-hot encode categories (object + pandas string/category)
    target_col = "fraud_bool"
    cat_cols = [
        c for c in train.columns
        if c != target_col and (
            pd.api.types.is_object_dtype(train[c])
            or pd.api.types.is_string_dtype(train[c])
            or pd.api.types.is_categorical_dtype(train[c])
        )
    ]
    if cat_cols:
        train = pd.get_dummies(train, columns=cat_cols, dummy_na=False)
        test = pd.get_dummies(test, columns=cat_cols, dummy_na=False)

    # safety: if any non-numeric column remains, cast to string then one-hot
    residual_cat_cols = [
        c for c in train.columns
        if c != target_col and not pd.api.types.is_numeric_dtype(train[c])
    ]
    if residual_cat_cols:
        for c in residual_cat_cols:
            train[c] = train[c].astype(str)
            test[c] = test[c].astype(str)
        train = pd.get_dummies(train, columns=residual_cat_cols, dummy_na=False)
        test = pd.get_dummies(test, columns=residual_cat_cols, dummy_na=False)

    y = train[target_col].astype(np.int8)
    X = train.drop(columns=[target_col], errors="ignore")
    X_test = test.copy()

    X, X_test = X.align(X_test, join="left", axis=1, fill_value=0)

    # final numeric safety before float cast
    for df in (X, X_test):
        non_num = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        for c in non_num:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        if non_num:
            df[non_num] = df[non_num].fillna(0)

    X = X.astype(np.float32)
    X_test = X_test.astype(np.float32)

    meta = {
        "seed": SEED,
        "n_features": int(X.shape[1]),
        "features": X.columns.tolist(),
        "dropped": drop_base,
        "cat_cols_encoded": cat_cols,
        "req_rate_special_features": [
            "feat_req24_over_4w", "feat_req6_over_24", "feat_req6_over_4w",
            "req_rate_4w_bin3", "req_rate_24h_bin2",
        ],
        "month_idx_treatment": "onehot(month_idx_cat)+sin_cos",
        "mi_top_binary_kept": mi_top_bin,
        "mi_device_email_flags": ["flag_device_email_neq1", "flag_device_email_missing"],
        "age_adjusted_columns": [
            "credit_risk_score", "req_credit_limit", "curr_addr_months", "name_email_sim", "session_length_min",
        ],
    }
    return X, y, X_test, meta


X, y, X_test, preprocess_meta = preprocess_guided(train_raw, test_raw)
print("X:", X.shape, "X_test:", X_test.shape)
print("fraud ratio:", y.mean())

X: (700000, 105) X_test: (300000, 105)
fraud ratio: 0.010917142857142858


## 4) 학습/검증 설정 및 평가 지표

In [4]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)


def ap_metric(y_true, y_prob):
    return average_precision_score(y_true, y_prob)


if len(X) > 150000:
    rs = np.random.RandomState(SEED)
    idx = rs.choice(len(X), 150000, replace=False)
    X_tune = X.iloc[idx].reset_index(drop=True)
    y_tune = y.iloc[idx].reset_index(drop=True)
else:
    X_tune = X.copy()
    y_tune = y.copy()

print("tuning sample:", X_tune.shape)
print("class ratio:", y_tune.mean())

tuning sample: (150000, 105)
class ratio: 0.011


In [5]:
# Safety cleanup for XGBoost: remove inf/overflow and fill NaN.
for df_name, df in [("X", X), ("X_test", X_test), ("X_tune", X_tune)]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.clip(lower=-1e12, upper=1e12, inplace=True)
    df.fillna(0.0, inplace=True)
    inf_cnt = int(np.isinf(df.to_numpy(dtype=np.float64)).sum())
    print(f"{df_name} inf count after sanitize: {inf_cnt}")

X inf count after sanitize: 0
X_test inf count after sanitize: 0
X_tune inf count after sanitize: 0


## 5) 3모델 튜닝/학습 (LightGBM, CatBoost, XGBoost) + 6) OOF 앙상블

In [ ]:
scale_pos = float((y == 0).sum() / max((y == 1).sum(), 1))


def build_repeated_cv(seed_offset: int = 0):
    return RepeatedStratifiedKFold(
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS_TUNE,
        random_state=SEED + seed_offset,
    )


def _cv_score_lgb(params, X_data, y_data):
    cv_rep = build_repeated_cv(11)
    scores = []
    for tr, va in cv_rep.split(X_data, y_data):
        m = lgb.LGBMClassifier(**params)
        m.fit(
            X_data.iloc[tr],
            y_data.iloc[tr],
            eval_set=[(X_data.iloc[va], y_data.iloc[va])],
            eval_metric="average_precision",
            callbacks=[lgb.early_stopping(80, verbose=False)],
        )
        p = m.predict_proba(X_data.iloc[va])[:, 1]
        scores.append(ap_metric(y_data.iloc[va], p))
    return float(np.mean(scores))


def _cv_score_xgb(params, X_data, y_data):
    cv_rep = build_repeated_cv(29)
    scores = []
    for tr, va in cv_rep.split(X_data, y_data):
        m = xgb.XGBClassifier(**params)
        m.fit(
            X_data.iloc[tr],
            y_data.iloc[tr],
            eval_set=[(X_data.iloc[va], y_data.iloc[va])],
            verbose=False,
        )
        p = m.predict_proba(X_data.iloc[va])[:, 1]
        scores.append(ap_metric(y_data.iloc[va], p))
    return float(np.mean(scores))


def _cv_score_cat(params, X_data, y_data):
    cv_rep = build_repeated_cv(47)
    scores = []
    for tr, va in cv_rep.split(X_data, y_data):
        m = CatBoostClassifier(**params)
        m.fit(
            X_data.iloc[tr],
            y_data.iloc[tr],
            eval_set=(X_data.iloc[va], y_data.iloc[va]),
            verbose=False,
        )
        p = m.predict_proba(X_data.iloc[va])[:, 1]
        scores.append(ap_metric(y_data.iloc[va], p))
    return float(np.mean(scores))


def lgb_objective(trial):
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 10),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 140),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.55, 0.98),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.55, 0.98),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 20.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 20.0, log=True),
        "scale_pos_weight": scale_pos,
        "n_estimators": trial.suggest_int("n_estimators", 700, 2200),
        "random_state": SEED,
        "n_jobs": -1,
        "verbose": -1,
    }
    return _cv_score_lgb(params, X_tune, y_tune)


def xgb_objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.12, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "n_estimators": trial.suggest_int("n_estimators", 700, 2600),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 120),
        "subsample": trial.suggest_float("subsample", 0.55, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
        "gamma": trial.suggest_float("gamma", 0.0, 3.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 30.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 30.0, log=True),
        "scale_pos_weight": scale_pos,
        "random_state": SEED,
        "n_jobs": -1,
        "tree_method": "hist",
    }
    return _cv_score_xgb(params, X_tune, y_tune)


def cat_objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 1000, 3200),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.12, log=True),
        "depth": trial.suggest_int("depth", 3, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 40.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.1, 8.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 8.0),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "bootstrap_type": "Bayesian",
        "loss_function": "Logloss",
        "eval_metric": "PRAUC",
        "random_seed": SEED,
        "auto_class_weights": "Balanced",
        "early_stopping_rounds": 100,
        "verbose": False,
    }
    return _cv_score_cat(params, X_tune, y_tune)


if USE_OPTUNA:
    sampler_lgb = optuna.samplers.TPESampler(seed=SEED + 101)
    sampler_xgb = optuna.samplers.TPESampler(seed=SEED + 202)
    sampler_cat = optuna.samplers.TPESampler(seed=SEED + 303)

    study_lgb = optuna.create_study(direction="maximize", sampler=sampler_lgb)
    study_xgb = optuna.create_study(direction="maximize", sampler=sampler_xgb)
    study_cat = optuna.create_study(direction="maximize", sampler=sampler_cat)

    study_lgb.optimize(lgb_objective, n_trials=N_TRIALS_PER_MODEL, show_progress_bar=True)
    study_xgb.optimize(xgb_objective, n_trials=N_TRIALS_PER_MODEL, show_progress_bar=True)
    study_cat.optimize(cat_objective, n_trials=N_TRIALS_PER_MODEL, show_progress_bar=True)

    best_lgb = study_lgb.best_trial.params.copy()
    best_lgb.update({
        "objective": "binary",
        "metric": "average_precision",
        "scale_pos_weight": scale_pos,
        "random_state": SEED,
        "n_jobs": -1,
        "verbose": -1,
    })

    best_xgb = study_xgb.best_trial.params.copy()
    best_xgb.update({
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "scale_pos_weight": scale_pos,
        "random_state": SEED,
        "n_jobs": -1,
        "tree_method": "hist",
    })

    best_cat = study_cat.best_trial.params.copy()
    best_cat.update({
        "bootstrap_type": "Bayesian",
        "loss_function": "Logloss",
        "eval_metric": "PRAUC",
        "random_seed": SEED,
        "auto_class_weights": "Balanced",
        "early_stopping_rounds": 100,
        "verbose": False,
    })
else:
    study_lgb = study_xgb = study_cat = None
    best_lgb = {
        "objective": "binary", "metric": "average_precision",
        "learning_rate": 0.017442857894326952,
        "num_leaves": 146, "max_depth": 2, "min_child_samples": 10,
        "feature_fraction": 0.7820006442886807, "bagging_fraction": 0.7102806714210247, "bagging_freq": 6,
        "reg_alpha": 0.005102117718947747, "reg_lambda": 0.007049146416643206,
        "scale_pos_weight": scale_pos,
        "n_estimators": 1200, "random_state": SEED, "n_jobs": -1, "verbose": -1,
    }
    best_xgb = {
        "objective": "binary:logistic", "eval_metric": "aucpr",
        "learning_rate": 0.020471986048315342,
        "max_depth": 3, "n_estimators": 1215, "min_child_weight": 8,
        "subsample": 0.6273251617739808, "colsample_bytree": 0.9112713371616585,
        "reg_alpha": 1.4359124369683893, "reg_lambda": 0.00011035517402844452,
        "scale_pos_weight": scale_pos,
        "random_state": SEED, "n_jobs": -1, "tree_method": "hist",
    }
    best_cat = {
        "iterations": 2000, "learning_rate": 0.040331443383617174, "depth": 4,
        "l2_leaf_reg": 8.234548958371454, "random_strength": 0.514960710230099,
        "border_count": 87, "subsample": 0.8233119185389446,
        "bootstrap_type": "Bernoulli", "loss_function": "Logloss", "eval_metric": "PRAUC",
        "random_seed": SEED, "auto_class_weights": "Balanced", "early_stopping_rounds": 80, "verbose": False,
    }

print("best_lgb:", best_lgb)
print("best_xgb:", best_xgb)
print("best_cat:", best_cat)

oof_lgb = np.zeros(len(X), dtype=np.float64)
oof_xgb = np.zeros(len(X), dtype=np.float64)
oof_cat = np.zeros(len(X), dtype=np.float64)

pred_lgb = np.zeros(len(X_test), dtype=np.float64)
pred_xgb = np.zeros(len(X_test), dtype=np.float64)
pred_cat = np.zeros(len(X_test), dtype=np.float64)

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    m_lgb = lgb.LGBMClassifier(**best_lgb)
    m_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="average_precision", callbacks=[lgb.early_stopping(100, verbose=False)])
    p_lgb = m_lgb.predict_proba(X_va)[:, 1]
    oof_lgb[va_idx] = p_lgb
    pred_lgb += m_lgb.predict_proba(X_test)[:, 1] / N_SPLITS

    m_xgb = xgb.XGBClassifier(**best_xgb)
    m_xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    p_xgb = m_xgb.predict_proba(X_va)[:, 1]
    oof_xgb[va_idx] = p_xgb
    pred_xgb += m_xgb.predict_proba(X_test)[:, 1] / N_SPLITS

    m_cat = CatBoostClassifier(**best_cat)
    m_cat.fit(X_tr, y_tr, eval_set=(X_va, y_va), verbose=False)
    p_cat = m_cat.predict_proba(X_va)[:, 1]
    oof_cat[va_idx] = p_cat
    pred_cat += m_cat.predict_proba(X_test)[:, 1] / N_SPLITS

    print(f"fold {fold} | LGB AP={ap_metric(y_va, p_lgb):.6f} | XGB AP={ap_metric(y_va, p_xgb):.6f} | CAT AP={ap_metric(y_va, p_cat):.6f}")

stack_oof = np.vstack([oof_lgb, oof_xgb, oof_cat]).T
stack_test = np.vstack([pred_lgb, pred_xgb, pred_cat]).T

def ens_objective(trial):
    w_l = trial.suggest_float("w_lgb", 0.0, 1.0)
    w_x = trial.suggest_float("w_xgb", 0.0, 1.0)
    w_c = trial.suggest_float("w_cat", 0.0, 1.0)
    w = np.array([w_l, w_x, w_c], dtype=np.float64)
    w = w / (w.sum() + 1e-12)
    pred = (stack_oof * w).sum(axis=1)
    return ap_metric(y, pred)

study_ens = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study_ens.optimize(ens_objective, n_trials=N_TRIALS_ENSEMBLE, show_progress_bar=False)

raw_w = np.array([
    study_ens.best_trial.params["w_lgb"],
    study_ens.best_trial.params["w_xgb"],
    study_ens.best_trial.params["w_cat"],
], dtype=np.float64)
ens_w = raw_w / (raw_w.sum() + 1e-12)

oof_ens = (stack_oof * ens_w).sum(axis=1)
test_pred = (stack_test * ens_w).sum(axis=1)

scores = {
    "lgb_ap": ap_metric(y, oof_lgb),
    "xgb_ap": ap_metric(y, oof_xgb),
    "cat_ap": ap_metric(y, oof_cat),
    "ens_ap": ap_metric(y, oof_ens),
    "lgb_auc": roc_auc_score(y, oof_lgb),
    "xgb_auc": roc_auc_score(y, oof_xgb),
    "cat_auc": roc_auc_score(y, oof_cat),
    "ens_auc": roc_auc_score(y, oof_ens),
}

print("ensemble weights [lgb, xgb, cat] =", ens_w)
print("scores:", {k: round(v, 6) for k, v in scores.items()})

[I 2026-03-28 02:24:14,994] A new study created in memory with name: no-name-8ef00454-b6b9-426b-b24f-5e8179b5fefa
[I 2026-03-28 02:24:14,996] A new study created in memory with name: no-name-ac6fd0b6-5b96-4076-96f4-c1a43b607c3b
[I 2026-03-28 02:24:14,996] A new study created in memory with name: no-name-2af1fccb-d580-49ae-a5d6-542c85fdd577


  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-03-28 02:24:57,787] Trial 0 finished with value: 0.14875464547400272 and parameters: {'learning_rate': 0.06147853910734938, 'num_leaves': 107, 'max_depth': 4, 'min_child_samples': 65, 'feature_fraction': 0.7943774339256552, 'bagging_fraction': 0.9289507690100379, 'bagging_freq': 1, 'reg_alpha': 0.00011131764680586144, 'reg_lambda': 0.0011380604360289523, 'n_estimators': 951}. Best is trial 0 with value: 0.14875464547400272.
[I 2026-03-28 02:25:49,419] Trial 1 finished with value: 0.056243843402671245 and parameters: {'learning_rate': 0.13498548339927013, 'num_leaves': 43, 'max_depth': 0, 'min_child_samples': 46, 'feature_fraction': 0.8119970668546448, 'bagging_fraction': 0.5803479796522939, 'bagging_freq': 6, 'reg_alpha': 0.00579048445109216, 'reg_lambda': 0.00243792467121291, 'n_estimators': 838}. Best is trial 0 with value: 0.14875464547400272.
[I 2026-03-28 02:27:36,784] Trial 2 finished with value: 0.1638650873857378 and parameters: {'learning_rate': 0.04748877021980542, 'n

  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-03-28 03:34:07,169] Trial 0 finished with value: 0.16483385600134415 and parameters: {'learning_rate': 0.01259408395055608, 'max_depth': 3, 'n_estimators': 1173, 'min_child_weight': 48, 'subsample': 0.5825771875718512, 'colsample_bytree': 0.6479401251089852, 'gamma': 2.9714528637400766, 'reg_alpha': 0.28593799949330195, 'reg_lambda': 0.3512665058799589}. Best is trial 0 with value: 0.16483385600134415.


## 7) 제출 파일/모델/리포트 저장

In [ ]:
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
sub_col = "fraud_bool" if "fraud_bool" in sample_sub.columns else sample_sub.columns[-1]

submission = sample_sub.copy()
submission[sub_col] = test_pred
submission_path = DATA_DIR / "submission_fraud_ensemble_same_preprocess.csv"
submission.to_csv(submission_path, index=False)

# full-data fit artifacts
final_lgb = lgb.LGBMClassifier(**best_lgb)
final_lgb.fit(X, y)
joblib.dump(final_lgb, DATA_DIR / "fraud_sameprep_lgb_model.joblib")

final_xgb = xgb.XGBClassifier(**best_xgb)
final_xgb.fit(X, y, verbose=False)
joblib.dump(final_xgb, DATA_DIR / "fraud_sameprep_xgb_model.joblib")

final_cat = CatBoostClassifier(**best_cat)
final_cat.fit(X, y, verbose=False)
joblib.dump(final_cat, DATA_DIR / "fraud_sameprep_cat_model.joblib")

report = {
    "seed": SEED,
    "target_ratio": float(y.mean()),
    "n_features": int(X.shape[1]),
    "scores": {k: float(v) for k, v in scores.items()},
    "ensemble_weights": {
        "lgb": float(ens_w[0]),
        "xgb": float(ens_w[1]),
        "cat": float(ens_w[2]),
    },
    "best_params": {
        "lgb": best_lgb,
        "xgb": best_xgb,
        "cat": best_cat,
    },
    "optuna_best_values": {
        "lgb": float(study_lgb.best_value) if study_lgb is not None else None,
        "xgb": float(study_xgb.best_value) if study_xgb is not None else None,
        "cat": float(study_cat.best_value) if study_cat is not None else None,
        "ensemble": float(study_ens.best_value),
    },
    "files": {
        "submission": str(submission_path),
        "lgb_model": "fraud_sameprep_lgb_model.joblib",
        "xgb_model": "fraud_sameprep_xgb_model.joblib",
        "cat_model": "fraud_sameprep_cat_model.joblib",
    },
}

with open(DATA_DIR / "fraud_ensemble_same_preprocess_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

with open(DATA_DIR / "fraud_ensemble_same_preprocess_meta.json", "w", encoding="utf-8") as f:
    json.dump(preprocess_meta, f, ensure_ascii=False, indent=2)

print("saved:")
print(" -", submission_path)
print(" - fraud_sameprep_lgb_model.joblib")
print(" - fraud_sameprep_xgb_model.joblib")
print(" - fraud_sameprep_cat_model.joblib")
print(" - fraud_ensemble_same_preprocess_report.json")
print(" - fraud_ensemble_same_preprocess_meta.json")

In [ ]:
# Full-train CatBoost (no Optuna, no KFold, no validation)
# Prerequisite: preprocess cell must already define X, y, X_test

best_cat_full = {
    "iterations": 2369,
    "learning_rate": 0.05156911709595948,
    "depth": 3,
    "l2_leaf_reg": 18.56686944170984,
    "random_strength": 3.125526917665803,
    "bagging_temperature": 2.1467430226178355,
    "border_count": 97,
    "bootstrap_type": "Bayesian",
    "loss_function": "Logloss",
    "eval_metric": "PRAUC",
    "random_seed": 42,
    "auto_class_weights": "Balanced",
    "verbose": False,
}

model_full = CatBoostClassifier(**best_cat_full)
model_full.fit(X, y, verbose=False)

test_pred_full = model_full.predict_proba(X_test)[:, 1]

sub_full = pd.read_csv(SAMPLE_SUB_PATH)
sub_full["probability"] = test_pred_full

out_path = DATA_DIR / "submission_catboost_fulltrain_no_cv.csv"
sub_full.to_csv(out_path, index=False)

joblib.dump(model_full, DATA_DIR / "catboost_fulltrain_no_cv.joblib")

print("saved submission:", out_path)
print("saved model:", DATA_DIR / "catboost_fulltrain_no_cv.joblib")
print(sub_full.head())